# CrashDiag base-Qwen evaluation on Kaggle

This independent, evaluation-only notebook measures the untrained `Qwen/Qwen2.5-3B-Instruct` base model against the exact signed 96-row evaluation split generated for a fresh run. It uses deterministic generation and executes each proposed JSON action in the CrashDiag environment. It does not run an agent loop, train weights, or use LLM judging.

In [ ]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import json
import os
import re
import subprocess
import sys

WORKFLOW_VERSION = "sft-qwen2.5_3b_instruct-evaluation-v1"
REPO_URL = "https://github.com/Indium-AI-Labs/CrashDiag.git"
REPO_DIR = Path("/kaggle/working/CrashDiag")
BUCKET_ID = "devaanshpa/CrashDiag"
SANDBOX_URL = "https://sandbox.devaanshpathak.com"
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
MODEL_SLUG = "qwen2.5_3b_instruct"
SFT_RUN_ID = os.environ.get("CRASHDIAG_SFT_RUN_ID") or "PASTE_SFT_RUN_ID_HERE"
DATASET_RUN_ID = os.environ.get("CRASHDIAG_DATASET_RUN_ID") or "PASTE_DATASET_RUN_ID_HERE"
DATASET_SOURCE_COMMIT = os.environ.get("CRASHDIAG_DATASET_SOURCE_COMMIT") or "PASTE_DATASET_SOURCE_COMMIT_HERE"
EVALUATOR_COMMIT = os.environ.get("CRASHDIAG_EVALUATOR_COMMIT") or "ddcd57f570989b597c1a2f27be2b4aaa6a368f25"
SFT_EVAL_RUN_ID = os.environ.get("CRASHDIAG_SFT_EVAL_RUN_ID") or (
    datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y%m%dT%H%M%SIST") + "-qwen2.5_3b_instruct-sft-eval"
)
SFT_EVAL_STAGE = "sft-evaluation"
PRECISION = "auto"
EXPECTED_ROWS = 96

for name, value in (("DATASET_SOURCE_COMMIT", DATASET_SOURCE_COMMIT), ("EVALUATOR_COMMIT", EVALUATOR_COMMIT)):
    if re.fullmatch(r"[0-9a-f]{40}", value) is None:
        raise ValueError(f"{name} must be a full lowercase Git SHA")
for name, value in (("DATASET_RUN_ID", DATASET_RUN_ID), ("SFT_RUN_ID", SFT_RUN_ID), ("SFT_EVAL_RUN_ID", SFT_EVAL_RUN_ID)):
    if value.startswith("PASTE_"):
        raise ValueError(f"Set {name} before continuing")
print(f"WORKFLOW_VERSION={WORKFLOW_VERSION}\nDATASET_RUN_ID={DATASET_RUN_ID}\nSFT_EVAL_RUN_ID={SFT_EVAL_RUN_ID}\nEVALUATOR_COMMIT={EVALUATOR_COMMIT}")

## Install the pinned evaluator

In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"Refusing to overwrite {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", EVALUATOR_COMMIT], check=True)
CURRENT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if CURRENT_COMMIT != EVALUATOR_COMMIT:
    raise RuntimeError("evaluator checkout mismatch")
torchao_probe = subprocess.run([sys.executable, "-m", "pip", "show", "torchao"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
if torchao_probe.returncode == 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train]"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"checked_out_evaluator={CURRENT_COMMIT}")

## Load Kaggle secrets

In [ ]:
def required_secret(name: str) -> str:
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise RuntimeError(f"Attach Kaggle Secret {name!r}") from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty")
    return value

os.environ["HF_TOKEN"] = required_secret("HF_TOKEN")
os.environ["CRASHDIAG_SANDBOX_TOKEN"] = required_secret("CRASHDIAG_SANDBOX_TOKEN")
os.environ["CRASHDIAG_SANDBOX_URL"] = SANDBOX_URL
os.environ["CRASHDIAG_HF_BUCKET_ID"] = BUCKET_ID
os.environ["CRASHDIAG_ARTIFACT_PREFIX"] = "runs"
os.environ["CRASHDIAG_ARTIFACT_LOCAL_ROOT"] = str(REPO_DIR / "artifacts")
os.environ["CRASHDIAG_ARTIFACT_UPLOAD_POLICY"] = "required"
os.environ["CRASHDIAG_RUN_ID"] = SFT_EVAL_RUN_ID
print("Kaggle Secrets loaded; values were not printed.")

## Download and verify the signed evaluation split

In [ ]:
from training.artifacts import ArtifactConfig, ArtifactUploader
from training.calibrate_grpo import read_jsonl

def make_uploader(run_id: str) -> ArtifactUploader:
    return ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=run_id, prefix="runs", policy="required", local_root=REPO_DIR / "artifacts", token=os.environ["HF_TOKEN"]))

def fetch_stage(client: ArtifactUploader, stage: str, target: Path, paths: list[str]) -> Path:
    if target.exists() and any(target.iterdir()):
        client.verify_local_stage(target, stage, include_paths=paths)
    else:
        client.download_stage(stage, target, include_paths=paths)
    return target

def remote_path_exists(client: ArtifactUploader, path: str) -> bool:
    return any(getattr(item, "path", None) == path for item in client.api.get_bucket_paths_info(client.config.bucket_id, [path]))

HANDOFF = Path("/kaggle/working/crashdiag-sft-model")
dataset_client = make_uploader(DATASET_RUN_ID)
DATASET_DIR = fetch_stage(dataset_client, "datasets", HANDOFF / DATASET_RUN_ID / "datasets", ["grpo_eval.jsonl"])
dataset_manifest = json.loads((DATASET_DIR / "manifest.json").read_text(encoding="utf-8"))
if dataset_manifest.get("runtime", {}).get("git_commit") != DATASET_SOURCE_COMMIT:
    raise RuntimeError("dataset/source commit mismatch")
EVAL_FILE = DATASET_DIR / "grpo_eval.jsonl"
if len(read_jsonl(EVAL_FILE)) != EXPECTED_ROWS:
    raise RuntimeError("evaluation split is not exactly 96 rows")
print(f"verified exact {EXPECTED_ROWS}-row split from {dataset_client.remote_uri('datasets')}")
sft_client = make_uploader(SFT_RUN_ID)
SFT_DIR = fetch_stage(sft_client, "sft", HANDOFF / SFT_RUN_ID / "sft", ["adapter_config.json", "adapter_model.safetensors", "tokenizer.json", "tokenizer_config.json"])
adapter_config = json.loads((SFT_DIR / "adapter_config.json").read_text(encoding="utf-8"))
if adapter_config.get("base_model_name_or_path") != BASE_MODEL:
    raise RuntimeError("SFT adapter/base-model mismatch")
print(f"verified signed SFT adapter from {sft_client.remote_uri('sft')}")

## Verify the live environment and GPU

In [ ]:
from crashdiag.sandbox_apps.http import HttpSandbox
import torch

with HttpSandbox(SANDBOX_URL, api_token=os.environ["CRASHDIAG_SANDBOX_TOKEN"], timeout=15.0) as sandbox:
    service = sandbox.service_health()
    application = sandbox.health_check()
if 2 not in service.get("scenario_schema_versions", []):
    raise RuntimeError(f"Vultr service lacks schema v2: {service}")
if service.get("hard_scenario_batch") is not True:
    raise RuntimeError("Vultr service lacks atomic hard-scenario setup")
if application.get("healthy") is not True:
    raise RuntimeError(f"sandbox preflight failed: {application}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU")
print(f"gpu={torch.cuda.get_device_name(0)}, bf16={torch.cuda.is_bf16_supported()}")

## Evaluate the base policy and sign the result

In [ ]:
import time
import training.evaluate_jsonl as _base_eval_module
from IPython.display import SVG, display
from training.evaluate_jsonl import main as evaluate_jsonl_main

# This wrapper observes mechanical results only; it does not alter generation or scoring.
if not hasattr(_base_eval_module, "_crashdiag_unwrapped_mechanical_reward"):
    _base_eval_module._crashdiag_unwrapped_mechanical_reward = _base_eval_module.mechanical_reward
_original_mechanical_reward = _base_eval_module._crashdiag_unwrapped_mechanical_reward
_eval_progress = {"completed": 0, "resolved": 0, "started": time.monotonic()}

def _mechanical_reward_with_progress(*args, **kwargs):
    rewards = _original_mechanical_reward(*args, **kwargs)
    _eval_progress["completed"] += len(rewards)
    _eval_progress["resolved"] += sum(float(reward) == 1.0 for reward in rewards)
    completed = int(_eval_progress["completed"])
    if completed == 1 or completed % 4 == 0 or completed >= EXPECTED_ROWS:
        elapsed = max(time.monotonic() - float(_eval_progress["started"]), 1e-6)
        rows_per_second = completed / elapsed
        eta_seconds = max(EXPECTED_ROWS - completed, 0) / rows_per_second
        fault_values = kwargs.get("fault_name") or []
        last_fault = str(fault_values[-1]) if fault_values else "unknown"
        gpu_gib = torch.cuda.memory_allocated() / (1024 ** 3)
        print(
            f"[sft-model-eval] {completed}/{EXPECTED_ROWS} "
            f"({completed / EXPECTED_ROWS:.1%}) resolved={_eval_progress['resolved']}/{completed} "
            f"elapsed={elapsed / 60:.1f}m eta={eta_seconds / 60:.1f}m "
            f"last_fault={last_fault} gpu_allocated={gpu_gib:.2f}GiB",
            flush=True,
        )
    return rewards

_base_eval_module.mechanical_reward = _mechanical_reward_with_progress
print(f"Progress reporting armed for {EXPECTED_ROWS} rows; model loading happens before the first row update.")

from IPython.display import SVG, display
from training.evaluate_jsonl import main as evaluate_jsonl_main

uploader = make_uploader(SFT_EVAL_RUN_ID)
BASE_QWEN_OUTPUT = REPO_DIR / "outputs/sft-evaluation"
BASE_QWEN_CACHE = HANDOFF / SFT_EVAL_RUN_ID / SFT_EVAL_STAGE
BASE_QWEN_PATHS = ["mechanical_evaluation.json", "reports/mechanical_evaluation_summary.json", "reports/mechanical_success_by_fault.svg"]
stage_complete = uploader.stage_is_complete(SFT_EVAL_STAGE)
run_complete = remote_path_exists(uploader, f"{uploader.config.remote_root}/_SUCCESS.json")
if run_complete and not stage_complete:
    raise RuntimeError("base-Qwen run is complete but its evaluation stage is missing")
if not run_complete and not stage_complete:
    uploader.start_run({"workflow": WORKFLOW_VERSION, "model": BASE_MODEL, "sft_run_id": SFT_RUN_ID, "dataset_run_id": DATASET_RUN_ID, "evaluator_commit": EVALUATOR_COMMIT, "scoring": "mechanical_fault_resolution"})

if stage_complete:
    BASE_QWEN_DIR = fetch_stage(uploader, SFT_EVAL_STAGE, BASE_QWEN_CACHE, BASE_QWEN_PATHS)
    print(f"reused signed stage {SFT_EVAL_STAGE}")
else:
    BASE_QWEN_DIR = BASE_QWEN_OUTPUT
    base_exit = evaluate_jsonl_main([
        "--model", str(SFT_DIR),
        "--dataset", str(EVAL_FILE),
        "--output-dir", str(BASE_QWEN_DIR),
        "--precision", PRECISION,
        "--max-new-tokens", "96",
        "--artifact-stage", SFT_EVAL_STAGE,
    ])
    if base_exit != 0:
        raise RuntimeError(f"base-Qwen evaluation failed with status {base_exit}")

base_report = json.loads((BASE_QWEN_DIR / "mechanical_evaluation.json").read_text(encoding="utf-8"))
summary = base_report["summary"]
if summary.get("total_episodes") != EXPECTED_ROWS:
    raise RuntimeError("base-Qwen report did not evaluate all rows")
if summary.get("backend_error_rate") != 0.0:
    raise RuntimeError("base-Qwen evaluation had sandbox backend errors")
if not run_complete:
    uploader.complete_run({"stages": [SFT_EVAL_STAGE], "workflow": WORKFLOW_VERSION, "model": BASE_MODEL, "sft_run_id": SFT_RUN_ID, "dataset_run_id": DATASET_RUN_ID, "evaluator_commit": EVALUATOR_COMMIT, "scoring": "mechanical_fault_resolution"})
print(json.dumps(summary, indent=2, sort_keys=True))
for chart in sorted((BASE_QWEN_DIR / "reports").glob("*.svg")):
    display(SVG(filename=str(chart)))
print(f"BASE_QWEN_ARTIFACTS={uploader.remote_uri()}")